# Assignment 4: Merging

## Submitting the assignment

1. When you are done, rename this Colab file to HW4 \<username\>, eg "HW4 deb219".
1. Then _**share**_ it with me (deb219@lehigh.edu) by clicking the big Share button in the upper right of this page.
1. And upload it to coursesite under HW4 for record keeping.

**I'll subtract points for not doing any these things.**

## Pre-requisites

- Normalized data lecture (08)
- Merging lecture (09), especially its Good Habits and Most Common Merging Parameters section

## Learning goals:

- Practice putting the merging habits to use

## Time expectations

**Stop working after 60 minutes and ask me and your classmates about it in the Groupme chat if you aren't done.**



## Part 1: Loading four datasets

I've done the work for you.

### Dataset 1: Compustat

Took this from lecture 06.

In [ ]:
# Load some data from here: https://github.com/LeDataSciFi/data
# Specifically, this is Compustat
# Pro-grade stuff

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# these three are used to download the file
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

url = 'https://github.com/LeDataSciFi/data/blob/main/Firm%20Year%20Datasets%20(Compustat)/CCM_cleaned_for_class.zip?raw=true'

#firms = pd.read_stata(url)
# <-- that code would work, but GH said it was too big and
# forced me to zip it, so here is the work around to download it:

with urlopen(url) as request:
    data = BytesIO(request.read())

with ZipFile(data) as archive:
    with archive.open(archive.namelist()[0]) as stata:
        firm_year_df = pd.read_stata(stata)

firm_year_df = firm_year_df.query('fyear < 2014') # partial coverage in this year

In [ ]:
firm_year_df.columns

Index(['gvkey', 'fyear', 'datadate', 'lpermno', 'gsector', 'sic', 'sic3',
       'age', 'tic', 'state', 'at', 'me', 'l_a', 'l_sale', 'prof_a', 'mb',
       'ppe_a', 'capx_a', 'xrd_a', 'cash_a', 'div_d', 'td', 'td_a', 'td_mv',
       'dltt_a', 'dv_a', 'invopps_FG09', 'sales_g', 'short_debt',
       'long_debt_dum', 'atr', 'smalltaxlosscarry', 'largetaxlosscarry',
       'tnic3hhi', 'tnic3tsimm', 'prodmktfluid', 'delaycon', 'equitydelaycon',
       'debtdelaycon', 'privdelaycon', 'l_emp', 'l_ppent', 'l_laborratio'],
      dtype='object')

In [ ]:
firm_year_df.shape

(222891, 43)

### Dataset 2: Firm founding

This is not the actual datasource I would use for this, but for now, we can use the wikipedia table.

In [ ]:
import requests
sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
response = requests.get(sp500_url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
firm_df = pd.read_html(response.text)[0]
firm_df

/tmp/ipython-input-3306078826.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  firm_df = pd.read_html(response.text)[0]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


### Dataset 3: Industry information

In [ ]:
# compute industry hhi from l_sale (log sales)
# HHI is the sum of the squares of the market shares of the firms in a market
# Market share is calculated as a firm's sales divided by the total sales in the industry
industry_year_df = (firm_year_df
                    .assign(sale = lambda x: np.exp(x['l_sale'])) # Convert log sales back to sales
                    .groupby(['sic3','fyear'])
                    .apply(lambda x: ((x['sale'] / x['sale'].sum())**2).sum()) # Calculate HHI
                    .to_frame(name='hhi') # Rename the resulting series to 'hhi'
)
display(industry_year_df)
industry_year_df.describe()

/tmp/ipython-input-3799441024.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ((x['sale'] / x['sale'].sum())**2).sum()) # Calculate HHI


hhi
sic3  fyear           
10.0  1975.0  0.337656
      1976.0  0.363268
      1977.0  0.359030
      1978.0  0.489423
      1979.0  0.438918
...                ...
999.0 2009.0  0.303575
      2010.0  0.301401
      2011.0  0.303151
      2012.0  0.334196
      2013.0  0.338807

[10424 rows x 1 columns]

,hhi
count,10424.000000
mean,0.384348
std,0.273772
min,0.000000
25%,0.175408
50%,0.299160
75%,0.519387
max,1.000000


### Dataset 4: Macro data

This dataset seems wrong.



In [ ]:
import pandas_datareader.data as web
yearly_df = web.DataReader(["GDP",'UNRATE','T10Y2Y'], "fred", start="1970-01-01")
yearly_df

,GDP,UNRATE,T10Y2Y
DATE,,,
1970-01-01,1051.200,3.9,NaN
1970-02-01,NaN,4.2,NaN
1970-03-01,NaN,4.4,NaN
1970-04-01,1067.375,4.6,NaN
1970-05-01,NaN,4.8,NaN
...,...,...,...
2025-09-18,NaN,NaN,0.54
2025-09-19,NaN,NaN,0.57
2025-09-22,NaN,NaN,0.54


## Part 2 - Planning and cleaning data before merges

The whole point here is that we want to ask some question that requires us to combine these four datasets. Assume that the Compustat dataset is the most important one, and we are trying to add new columns to that dataset.

### Exercise 1: Modify the code that gets the macro data above

That's not yearly data! It's daily!

And I don't want GDP levels, I want GDP growth rates for my analysis!

And,    
> 1. I prefer to make sure identifying (`on`) variables are not the index: `df = df.reset_index()`
> 1. Name the variables you might merge `on` the same in both datasets.
> 1. Bonus: Make sure any date variables you might use are formatted the same in both datasets (e.g. as years)

Change the code it so that it gets some kind of annual (once a year) version of each of those variables (you decide how to do this, but be thoughtful about it).

In [ ]:
# explain here why you made the changes you did to end up with your yearly_df

### Exercise 2:

Clean up any of the other datasets before your merges.

In [ ]:
# work here

### Exercise 3

Q: What are the observation units in each of these datasets? How do you know?

A: You know because I gave them good names!

- yearly_df units: year (if you changed it right)
- industry_year_df units: industry-year, so auto-2000, auto-2001, finance-2000, and so on. "Industry" is based on some variable called sic3, which is explained here: https://siccode.com/page/what-is-a-sic-code.



In [ ]:
# work here

## Part 3: Combining the data

The whole point here is that we want to ask some question

### Exercise 4: Combine all four datasets

Call the new dataset `analysis_firm_year_df`.

In [ ]:
# work here


In [ ]:
# leave this - if your merges are correct, run this, and you won't see and error
assert len(analysis_firm_year_df) == len(firm_year_df)

## Part 4: Analysis

What variables are most related to HHI (industry concentration)?
1. Start by looking at the variables we have. What are the top 5 highest correlations?
1. But you should go back and get more macro variables to try to find things that predict it: Ask a friend what macro variables might be related to HHI, and then get them and check.